## Linear operators
Import libraries

In [ ]:
import os
import sys
import numpy as np
import scipy as sp
import tensorflow as tf
sys.path.append("/home/zanardi/Codes/ML/PyCOMET/pycomet/")

from tqdm import tqdm
from silx.io.dictdump import h5todict
from pycomet.networks.chemkin.combustion.meno.blocks import POLLUOperators

In [ ]:
import utils

Define inputs

In [ ]:
# Paths
path_to_data = "./../00.datagen/data/"
paths = {
  "data": path_to_data + "/test/",
  "saving": "./output/lin_ops/"
}
os.makedirs(paths["saving"], exist_ok=True)
# Delta-A
# > Confidence interval
alpha = 0.95
uncert_name = f"ci_{int(100*alpha)}" if (alpha > 0) else "std"

In [ ]:
# Time grid
tlim = [1e-7, 60.0]
tdim = 200
t = np.geomspace(*tlim, num=tdim)

In [ ]:
indices = h5todict(path_to_data+"/physics/species_indices.hdf5")
indices = {
  "t": np.array([20]),
  "nlin": indices["nlin"]+21
}

Load data and model

In [ ]:
data = utils.load_test_cases(paths["data"])

Initialize the linear operators constructor

In [ ]:
linops = POLLUOperators(
  rates=path_to_data + "/physics/rates.hdf5",
  species_indices=path_to_data + "/physics/species_indices.hdf5",
  use_corr=False,
  scale_corr=1.0
)

Compute linear operator variability

In [ ]:
def compute_delta(t, y, teval=None, teps=1e-8):
  t = t.reshape(-1)
  axes = tuple(np.arange(y.ndim)[:-1].tolist())
  dydt = np.gradient(y, t, axis=-1, edge_order=2)
  delta = np.linalg.norm(dydt, axis=axes)
  delta /= (np.linalg.norm(y/(t+teps), axis=axes))
  if (teval is not None):
    delta = sp.interpolate.interp1d(t, delta, kind="cubic")(teval)
  return 100 * delta

In [ ]:
deltas = {"A": [], "b": []}
for (icase, idata) in tqdm(data.items()):
  idata = idata.to_numpy()
  ti, yi_nl = [np.take(idata, indices[k], axis=1) for k in ("t", "nlin")]
  Ai, bi = tf.nest.map_structure(lambda x: x.numpy(), linops((yi_nl, None)))
  deltas["A"].append(compute_delta(ti, Ai.T, t))
  deltas["b"].append(compute_delta(ti, bi.T, t))
deltas = {k: np.vstack(d) for (k, d) in deltas.items()}
deltas = tf.nest.map_structure(
  lambda x: utils.compute_stats(x, axis=0, alpha=alpha),
  deltas
)

In [ ]:
for k in ("A", "b"):
  utils.plot_err_continuous(
    x=t,
    mean=deltas[k][0],
    uncert=deltas[k][1],
    alpha=alpha,
    labels=[r"$t$ [s]", r"$\Delta\mathbf{%s}$" % (k,) + " [\%]"],
    scales=["log", "linear"],
    legend_loc="best",
    figname=paths["saving"]+ f"/delta_{k}.pdf",
    save=True,
    show=False
  )